# 实验 7 · 案例 2：唐诗信息爬取

使用 Scrapy 先解析诗词列表中的**标题、作者和详情链接**，再通过 `Request` 回调解析详情页正文，最后由 Pipeline 保存为 CSV。示例网址沿用教材中的 `bspider.top/gushiwen/ts`。

## 1. 环境与项目创建

In [1]:
import importlib.util
import subprocess
import sys

missing = [p for p in ("scrapy", "pandas") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

In [2]:
from pathlib import Path

project = Path.cwd() / "shici"
if not (project / "scrapy.cfg").exists():
    subprocess.run(
        [sys.executable, "-m", "scrapy", "startproject", "shici", str(project)],
        check=True,
    )
print(project)

c:\Users\ZhuanZ（无密码）\Desktop\BigDataCourse\大数据采集与处理-学生实验环境\workspace\bigdata-collection-labs-20261017\大数据采集与处理实验报告-7-邢贺-112412060139\shici


## 2. 定义 Item

In [3]:
%%writefile shici/shici/items.py
import scrapy


class ShiciItem(scrapy.Item):
    name = scrapy.Field()
    author = scrapy.Field()
    content = scrapy.Field()

Overwriting shici/shici/items.py


## 3. 编写 Spider：列表页 → 详情页

In [4]:
%%writefile shici/shici/spiders/tangshi.py
import re
import scrapy
from shici.items import ShiciItem


class TangshiSpider(scrapy.Spider):
    name = "tangshi"
    allowed_domains = ["www.bspider.top", "bspider.top"]
    start_urls = ["http://www.bspider.top/gushiwen/ts"]

    def parse(self, response):
        # 列表页：定位每首诗所在的 span
        for row in response.xpath('//div[@class="typecont"]/span'):
            href = row.xpath('a/@href').get()
            if not href:
                continue

            item = ShiciItem()
            item["name"] = row.xpath('a/text()').get(default="").strip()
            author = "".join(row.xpath('text()').getall())
            item["author"] = re.sub(r"[()（）\s]", "", author)

            # 把列表页数据传入详情页回调
            yield response.follow(
                href, callback=self.parse_detail, meta={"item": item}
            )

    def parse_detail(self, response):
        item = response.meta["item"]

        if not item["author"]:
            item["author"] = response.css("p.source a:first-child::text").get(default="").strip()

        # string() 合并正文及 br 分隔的文本
        content = response.xpath('string(//div[@class="contson"])').get(default="")
        item["content"] = re.sub(r"\s+", "", content)
        yield item

Overwriting shici/shici/spiders/tangshi.py


## 4. Pipeline：保存 CSV

In [5]:
%%writefile shici/shici/pipelines.py
import csv
from pathlib import Path
from itemadapter import ItemAdapter


class CsvPipeline:
    def open_spider(self, spider=None):
        self.file = (Path.cwd() / "tangshi.csv").open(
            "w", encoding="utf-8-sig", newline=""
        )
        self.writer = csv.DictWriter(
            self.file, fieldnames=["name", "author", "content"]
        )
        self.writer.writeheader()

    def process_item(self, item, spider=None):
        self.writer.writerow(ItemAdapter(item).asdict())
        return item

    def close_spider(self, spider=None):
        self.file.close()

Overwriting shici/shici/pipelines.py


## 5. 配置与启动

In [6]:
%%writefile shici/shici/settings.py
BOT_NAME = "shici"
SPIDER_MODULES = ["shici.spiders"]
NEWSPIDER_MODULE = "shici.spiders"

ROBOTSTXT_OBEY = True
DOWNLOAD_DELAY = 0.5
DOWNLOAD_TIMEOUT = 30
RETRY_TIMES = 5
LOG_LEVEL = "INFO"

ITEM_PIPELINES = {"shici.pipelines.CsvPipeline": 300}

Overwriting shici/shici/settings.py


In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "scrapy", "crawl", "tangshi"],
    cwd=project,
    text=True,
    encoding="utf-8",
    errors="replace",
)
result.check_returncode()

## 6. 查看采集结果

In [ ]:
import pandas as pd

csv_path = project / "tangshi.csv"
df = pd.read_csv(csv_path)
if df.empty:
    raise RuntimeError("No poems collected. Check the crawl log for HTTP or network errors and rerun the crawl cell.")
if df[["name", "author", "content"]].isna().any().any():
    raise RuntimeError("Some poem fields are missing. Check the detail page parsing and crawl log.")
display(df.head(10))
print("记录数：", len(df))

,name,author,content
0,赠孟浩然,李白,吾爱孟夫子，风流天下闻。红颜弃轩冕，白首卧松云。醉月频中圣，迷花不事君。高山安可仰，徒此揖清芬。
1,寄左省杜拾遗,岑参,联步趋丹陛，分曹限紫微。晓随天仗入，暮惹御香归。白发悲花落，青云羡鸟飞。圣朝无阙事，自觉谏书稀。
2,夜泊牛渚怀古,李白,牛渚西江夜，青天无片云。登舟望秋月，空忆谢将军。余亦能高咏，斯人不可闻。明朝挂帆席，枫叶落纷...
3,赋得古原草送别,白居易,离离原上草，一岁一枯荣。野火烧不尽，春风吹又生。远芳侵古道，晴翠接荒城。又送王孙去，萋萋满别情。
4,听蜀僧浚弹琴,李白,蜀僧抱绿绮，西下峨眉峰。为我一挥手，如听万壑松。客心洗流水，余响入霜钟。不觉碧山暮，秋云暗几重。
5,题大庾岭北驿,宋之问,阳月南飞雁，传闻至此回。我行殊未已，何日复归来。江静潮初落，林昏瘴不开。明朝望乡处，应见陇头梅。
6,贼平后送人北归,司空曙,世乱同南去，时清独北还。他乡生白发，旧国见青山。晓月过残垒，繁星宿故关。寒禽与衰草，处处伴愁颜。
7,喜外弟卢纶见宿,司空曙,静夜四无邻，荒居旧业贫。雨中黄叶树，灯下白头人。以我独沉久，愧君相见频。平生自有分，况是蔡家亲。
8,云阳馆与韩绅宿别,司空曙,故人江海别，几度隔山川。乍见翻疑梦，相悲各问年。孤灯寒照雨，深竹暗浮烟。(深竹一作：湿竹)更...
9,次北固山下,王湾,客路青山外，行舟绿水前。(青山外一作：青山下)潮平两岸阔，风正一帆悬。海日生残夜，江春入旧年...


记录数： 320
